In [ ]:
import joblib
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import re

MODEL_DIR = "all-mpnet-base-v2"
ROUTER_DIR = "trained_routers/trained_svr_regressor.joblib"
ROUTING_THRESHOLD = 0.6

encoder = SentenceTransformer(MODEL_DIR, device="cuda")

find_encoder = SentenceTransformer("all-roberta-large-v1", device="cuda")

router_model = joblib.load(ROUTER_DIR)
find_big_model = joblib.load("trained_routers/high_cost_classifier.joblib")

# Load pre-merged datasets from final_regressor
test_data_org = pd.read_csv("dataset/final_regressor/test.csv")
val_data_org = pd.read_csv("dataset/final_regressor/val.csv")


SIMPLICITY_WEIGHTS = {
    "is_correct_qwen_0_5b": 0.2,
    "is_correct_qwen_1_5b": 0.2,
    "is_correct_qwen_3b": 0.2,
    "is_correct_qwen_7b": 0.2,
    "is_correct_qwen_14b": 0.2,
}

def add_simplicity_score(df):
    missing = [col for col in SIMPLICITY_WEIGHTS if col not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns for simplicity_score: {missing}")

    score = sum(
        pd.to_numeric(df[col], errors="coerce").fillna(0).clip(lower=0, upper=1) * weight
        for col, weight in SIMPLICITY_WEIGHTS.items()
    )
    df["simplicity_score"] = 1 - score.clip(lower=0.0, upper=1.0)
    return df

test_data = add_simplicity_score(test_data_org)
val_data = add_simplicity_score(val_data_org)

X_val = encoder.encode(val_data["question"].tolist(), show_progress_bar=True)
y_val = val_data["simplicity_score"].values
X_test = encoder.encode(test_data["question"].tolist(), show_progress_bar=True)
y_test = test_data["simplicity_score"].values

# Clip predictions to [0, 1] range to avoid out-of-bounds issues in routing.
y_pred_score_val = router_model.predict(X_val).clip(0.0, 1.0)
y_pred_val = (y_pred_score_val <= ROUTING_THRESHOLD).astype(int)

true_simple = ((y_val <= ROUTING_THRESHOLD))
true_complex = ((y_val > ROUTING_THRESHOLD))
pred_simple = (y_pred_val == 1)
pred_complex = (y_pred_val == 0)

p_simple_simple = int((true_simple & pred_simple).sum())
p_simple_complex = int((true_simple & pred_complex).sum())
p_complex_simple = int((true_complex & pred_simple).sum())
p_complex_complex = int((true_complex & pred_complex).sum())

print("Validation complexity-label breakdown:")
print(f"true simple -> predicted simple: {p_simple_simple}")
print(f"true simple -> predicted complex: {p_simple_complex}")
print(f"true complex -> predicted simple: {p_complex_simple}")
print(f"true complex -> predicted complex: {p_complex_complex}")

# Configure selected models here.
SLM_MODEL = "qwen_7b"
LLM_MODEL = "gpt5"

# In this notebook, p comes from a regressor, so we derive class masks from a threshold.

df_val = val_data
df_test = test_data
pred_simple = (y_pred_score_val <= ROUTING_THRESHOLD)
pred_complex = (y_pred_score_val > ROUTING_THRESHOLD)
true_simple = (y_val <= ROUTING_THRESHOLD)
true_complex = (y_val > ROUTING_THRESHOLD)

def _model_cols(model_name):
    return {
        "correct": f"is_correct_{model_name}",
        "latency": f"latency_{model_name}",
        "input_tokens": f"input_tokens_{model_name}",
        "output_tokens": f"output_tokens_{model_name}",
    }

def _validate_model_cols(df, model_name):
    cols = _model_cols(model_name)
    missing = [col for col in cols.values() if col not in df.columns]
    if missing:
        raise KeyError(f"Missing columns for model '{model_name}': {missing}")
    return cols

slm_cols = _validate_model_cols(df_val, SLM_MODEL)
llm_cols = _validate_model_cols(df_val, LLM_MODEL)

slm_can = (pd.to_numeric(df_val[slm_cols["correct"]], errors="coerce").fillna(0).to_numpy() > 0.5).astype(int)
llm_can = (pd.to_numeric(df_val[llm_cols["correct"]], errors="coerce").fillna(0).to_numpy() > 0.5).astype(int)

print("\nOf predicted simple:")
print(f"SLM ({SLM_MODEL}) can solve: {int(slm_can[pred_simple].sum())} / {int(pred_simple.sum())}")
print(f"LLM ({LLM_MODEL}) can solve: {int(llm_can[pred_simple].sum())} / {int(pred_simple.sum())}")

print("\nOf predicted complex:")
print(f"SLM ({SLM_MODEL}) can solve: {int(slm_can[pred_complex].sum())} / {int(pred_complex.sum())}")
print(f"LLM ({LLM_MODEL}) can solve: {int(llm_can[pred_complex].sum())} / {int(pred_complex.sum())}")

q_s_complex = int(slm_can[pred_complex].sum()) / int(pred_complex.sum()) if pred_complex.sum() > 0 else 0
q_s_simple = int(slm_can[pred_simple].sum()) / int(pred_simple.sum()) if pred_simple.sum() > 0 else 0
q_l_complex = int(llm_can[pred_complex].sum()) / int(pred_complex.sum()) if pred_complex.sum() > 0 else 0
q_l_simple = int(llm_can[pred_simple].sum()) / int(pred_simple.sum()) if pred_simple.sum() > 0 else 0


print("\nEstimated probabilities:")
print(f"Q(SLM solves | predicted complex) = {q_s_complex:.2f}")
print(f"Q(SLM solves | predicted simple) = {q_s_simple:.2f}")
print(f"Q(LLM solves | predicted complex) = {q_l_complex:.2f}")
print(f"Q(LLM solves | predicted simple) = {q_l_simple:.2f}")

# Averages for the exact four probability scenarios above.
def _cond_avg(df, mask, model_cols):
    mask = np.asarray(mask, dtype=bool)
    if len(mask) != len(df):
        raise ValueError(f"Mask length {len(mask)} does not match dataframe length {len(df)}")
    n = int(mask.sum())
    if n == 0:
        return n, float("nan"), float("nan"), float("nan")
    avg_latency = pd.to_numeric(df.loc[mask, model_cols["latency"]], errors="coerce").mean()
    avg_input = pd.to_numeric(df.loc[mask, model_cols["input_tokens"]], errors="coerce").mean()
    avg_output = pd.to_numeric(df.loc[mask, model_cols["output_tokens"]], errors="coerce").mean()
    return n, avg_latency, avg_input, avg_output

llm_can_bool = llm_can.astype(bool)
slm_can_bool = slm_can.astype(bool)

masks = [
    ("Q(SLM solves | predicted complex)", pred_complex & slm_can_bool, slm_cols),
    ("Q(SLM solves | predicted simple)", pred_simple & slm_can_bool, slm_cols),
    ("Q(LLM solves | predicted complex)", pred_complex & llm_can_bool, llm_cols),
    ("Q(LLM solves | predicted simple)", pred_simple & llm_can_bool, llm_cols),
]

q_s_complex_metrics = _cond_avg(df_val, pred_complex & slm_can_bool, slm_cols)
q_s_simple_metrics = _cond_avg(df_val, pred_simple & slm_can_bool, slm_cols)
q_l_complex_metrics = _cond_avg(df_val, pred_complex & llm_can_bool, llm_cols)
q_l_simple_metrics = _cond_avg(df_val, pred_simple & llm_can_bool, llm_cols)

# Compute average input/output tokens and latency for the total of selected llm and slm.
slm_latency = pd.to_numeric(df_val[slm_cols["latency"]], errors="coerce").mean()
slm_input_tokens = pd.to_numeric(df_val[slm_cols["input_tokens"]], errors="coerce").mean()
slm_output_tokens = pd.to_numeric(df_val[slm_cols["output_tokens"]], errors="coerce").mean()
llm_latency = pd.to_numeric(df_val[llm_cols["latency"]], errors="coerce").mean()
llm_input_tokens = pd.to_numeric(df_val[llm_cols["input_tokens"]], errors="coerce").mean()
llm_output_tokens = pd.to_numeric(df_val[llm_cols["output_tokens"]], errors="coerce").mean()

print("\nDetailed metrics for each scenario:")
print(f"SLM avg latency: {slm_latency:.4f} seconds")
print(f"LLM avg latency: {llm_latency:.4f} seconds")
print(f"SLM avg input tokens: {slm_input_tokens:.4f}")
print(f"LLM avg input tokens: {llm_input_tokens:.4f}")
print(f"SLM avg output tokens: {slm_output_tokens:.4f}")
print(f"LLM avg output tokens: {llm_output_tokens:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix

GPT5_INPUT_PRICE_PER_1M = 1.25
GPT5_OUTPUT_PRICE_PER_1M = 10.0

# p = P(complex | x) is produced by the classifier.
# E[C | a, x] = c_a + lambda * t_a + C_err * (1 - q_a(x))
LAMBDA_LATENCY = 0.13
MU_COST = 100 # Make the units from dollars to cents
C_ERR = 14

MANUAL_Q_S_SIMPLE = q_s_simple
MANUAL_Q_S_COMPLEX = q_s_complex
MANUAL_Q_L_SIMPLE = q_l_simple
MANUAL_Q_L_COMPLEX = q_l_complex

def adjust_probability_llm(p, features):
    logit = np.log((p + 1e-8) / (1 - p + 1e-8))

    if features["has_indexed_vars"]:
        logit -= 1.2
    
    else:
        return p

    return 1 / (1 + np.exp(-logit))

def adjust_probability_slm(p, features):
    logit = np.log((p + 1e-8) / (1 - p + 1e-8))

    if features["is_simple_natural_language"]:
        logit +=0.5
    elif features["is_low_input"]:
        logit +=0.6
    else:
        return p

    return 1 / (1 + np.exp(-logit))

def error_cost(cost, latency, q, lambd=LAMBDA_LATENCY, mu=MU_COST, c_err=C_ERR):
    return cost*mu + lambd * latency + c_err * (1 - q)

def compute_expected_cost(input_tokens, output_tokens):
    total_cost = (input_tokens / 1_000_000) * GPT5_INPUT_PRICE_PER_1M + (output_tokens / 1_000_000) * GPT5_OUTPUT_PRICE_PER_1M
    # print(total_cost)
    return total_cost

def compute_qs(p):
    qs = (1-p) * MANUAL_Q_S_SIMPLE + p * MANUAL_Q_S_COMPLEX
    return qs

def compute_ql(p):
    ql = (1-p) * MANUAL_Q_L_SIMPLE + p * MANUAL_Q_L_COMPLEX
    return ql

def avg_ql():
    total = int(pred_complex.sum()) + int(pred_simple.sum())
    if total == 0:
        return 0.0
    return (int(pred_complex.sum()) * q_l_complex + int(pred_simple.sum()) * q_l_simple) / total

def avg_qs():
    total = int(pred_complex.sum()) + int(pred_simple.sum())
    if total == 0:
        return 0.0
    return (int(pred_complex.sum()) * q_s_complex + int(pred_simple.sum()) * q_s_simple) / total
    

def bayesian_decision_2path(p, features, lambd=LAMBDA_LATENCY, mu=MU_COST, cerr=C_ERR):
    p = float(np.asarray(p).reshape(-1)[-1])
    p_l = adjust_probability_llm(p, features)
    p_s = adjust_probability_slm(p, features)
    total_cost_small = 0
    total_cost_big = compute_expected_cost(llm_input_tokens, llm_output_tokens)
    ql = compute_ql(p_l)
    qs = compute_qs(p_s)
    error_big = error_cost(total_cost_big, llm_latency, ql, lambd=lambd, mu=mu, c_err=cerr)
    error_small = error_cost(total_cost_small, slm_latency, qs, lambd=lambd, mu=mu, c_err=cerr)
    if error_big < error_small:
        return "big"
    else:
        return "small"

def compute_threshold(C_ERR):
    delta_c = (compute_expected_cost(llm_input_tokens, llm_output_tokens) - 0) * MU_COST + LAMBDA_LATENCY * (llm_latency - slm_latency)
    d_simple = q_l_simple - q_s_simple
    d_complex = q_l_complex - q_s_complex
    return ((delta_c / C_ERR) - d_simple) / (d_complex - d_simple)

threshhold = compute_threshold(C_ERR)

print(f"Computed threshold: {threshhold:.4f}")

In [ ]:
# Decision loop using the existing bayesian_decision_2path from the previous cell
router_performance_data = []
best_route_performance_data = []

if "SLM_MODEL" not in globals():
    SLM_MODEL = "qwen_7b"
if "LLM_MODEL" not in globals():
    LLM_MODEL = "gpt5"

def _model_cols(model_name):
    return {
        "correct": f"is_correct_{model_name}",
        "latency": f"latency_{model_name}",
        "input_tokens": f"input_tokens_{model_name}",
        "output_tokens": f"output_tokens_{model_name}",
    }

def _to_bool(x):
    x_num = pd.to_numeric(pd.Series([x]), errors="coerce").iloc[0]
    return bool(pd.notna(x_num) and float(x_num) > 0.5)

def has_indexed_vars(text: str) -> bool:
    """
    Math signal:
    Detects indexed variables such as a_n, x_1, z_k, b_23.
    """
    if not isinstance(text, str):
        return False

    pattern = r"\b[a-zA-Z]_[a-zA-Z0-9]+\b"
    return bool(re.search(pattern, text))

def is_simple_natural_language(text):
    if len(text.split()) > 20:
        return False
    if any(c in text for c in ["_", "^", "\\", "{", "}"]):
        return False
    return True


slm_cols = _model_cols(SLM_MODEL)
llm_cols = _model_cols(LLM_MODEL)
required_cols = set(slm_cols.values()) | set(llm_cols.values())
missing_cols = sorted([c for c in required_cols if c not in df_test.columns])
if missing_cols:
    raise KeyError(f"Missing required test columns: {missing_cols}")

df_test_eval = df_test.reset_index(drop=True).copy()
test_probs = router_model.predict(X_test)

for data in [router_performance_data, best_route_performance_data]:
    for i, row in df_test_eval.iterrows():
        qid = row["id"] if "id" in row.index else i
        p_prob = float(test_probs[i])

        slm_correct_row = int(_to_bool(row[slm_cols["correct"]]))
        llm_correct_row = int(_to_bool(row[llm_cols["correct"]]))

        if data is router_performance_data:
            # features = {"has_indexed_vars": has_indexed_vars(row.get("text", "")),
            #             "is_simple_natural_language": is_simple_natural_language(row.get("text", "")),
            #             "is_low_input": pd.to_numeric(pd.Series([row[slm_cols["input_tokens"]]]), errors="coerce").iloc[0] < 12}
            
            features = {"has_indexed_vars": False,
                        "is_simple_natural_language": False,
                        "is_low_input": False}
            
            # big_text = find_encoder.encode([row.get("question", "")], normalize_embeddings=True, show_progress_bar=False)
            # big_decision = find_big_model.predict(big_text)
            # if big_decision == 1:
            #     decision = "big"
            # else:
            decision = bayesian_decision_2path(p_prob, features)
        else:
            slm_latency_row = pd.to_numeric(pd.Series([row[slm_cols["latency"]]]), errors="coerce").iloc[0]
            llm_latency_row = pd.to_numeric(pd.Series([row[llm_cols["latency"]]]), errors="coerce").iloc[0]
            llm_input_tokens_row = pd.to_numeric(pd.Series([row[llm_cols["input_tokens"]]]), errors="coerce").iloc[0]
            llm_output_tokens_row = pd.to_numeric(pd.Series([row[llm_cols["output_tokens"]]]), errors="coerce").iloc[0]
            small_error = error_cost(0.0, slm_latency_row, slm_correct_row)
            big_error = error_cost(compute_expected_cost(llm_input_tokens_row, llm_output_tokens_row), llm_latency_row, llm_correct_row)
            decision = "big" if big_error < small_error else "small"

        slm_latency_row = pd.to_numeric(pd.Series([row[slm_cols["latency"]]]), errors="coerce").iloc[0]
        slm_input_tokens_row = pd.to_numeric(pd.Series([row[slm_cols["input_tokens"]]]), errors="coerce").iloc[0]
        slm_output_tokens_row = pd.to_numeric(pd.Series([row[slm_cols["output_tokens"]]]), errors="coerce").iloc[0]
        llm_latency_row = pd.to_numeric(pd.Series([row[llm_cols["latency"]]]), errors="coerce").iloc[0]
        llm_input_tokens_row = pd.to_numeric(pd.Series([row[llm_cols["input_tokens"]]]), errors="coerce").iloc[0]
        llm_output_tokens_row = pd.to_numeric(pd.Series([row[llm_cols["output_tokens"]]]), errors="coerce").iloc[0]

        if decision == "small":
            latency = slm_latency_row
            in_tokens = slm_input_tokens_row
            out_tokens = slm_output_tokens_row
            correct_answer = slm_correct_row
            actual_cost = 0.0
            error = error_cost(actual_cost, latency, correct_answer)
            predicted_cost = 0.0
            predicted_latency = slm_latency
            router_q = compute_qs(p_prob)
        else:
            latency = llm_latency_row
            in_tokens = llm_input_tokens_row
            out_tokens = llm_output_tokens_row
            correct_answer = llm_correct_row
            actual_cost = compute_expected_cost(in_tokens, out_tokens)
            error = error_cost(actual_cost, latency, correct_answer)
            predicted_cost = compute_expected_cost(llm_input_tokens, llm_output_tokens)
            predicted_latency = llm_latency
            router_q = compute_ql(p_prob)

        error_arr = np.asarray(error).reshape(-1)
        error_scalar = error_arr[0] if error_arr.size > 0 else np.nan

        data.append({
            "id": qid,
            "p": p_prob,
            "decision": decision,
            "correct_answer": int(correct_answer),
            "total_latency": float(latency) if pd.notna(latency) else float("nan"),
            "input_tokens": float(in_tokens) if pd.notna(in_tokens) else float("nan"),
            "output_tokens": float(out_tokens) if pd.notna(out_tokens) else float("nan"),
            "actual_cost": float(actual_cost) if pd.notna(actual_cost) else float("nan"),
            "predicted_cost": float(predicted_cost) if pd.notna(predicted_cost) else float("nan"),
            "predicted_latency": float(predicted_latency) if pd.notna(predicted_latency) else float("nan"),
            "error_cost": float(error_scalar) if pd.notna(error_scalar) else float("nan"),
            "router_q": float(router_q) if pd.notna(router_q) else float("nan"),
        })

# Compare avg performance of router to small and big performance
# Skip rows where predicted big by find_big_model to isolate bayesian decision performance.
avg_latency_slm = pd.to_numeric(df_test_eval[slm_cols["latency"]], errors="coerce").mean()
avg_input_tokens_slm = pd.to_numeric(df_test_eval[slm_cols["input_tokens"]], errors="coerce").mean()
avg_output_tokens_slm = pd.to_numeric(df_test_eval[slm_cols["output_tokens"]], errors="coerce").mean()
avg_predicted_error_slm = error_cost(0.0, slm_latency, avg_qs())


avg_latency_llm = pd.to_numeric(df_test_eval[llm_cols["latency"]], errors="coerce").mean()
avg_input_tokens_llm = pd.to_numeric(df_test_eval[llm_cols["input_tokens"]], errors="coerce").mean()
avg_output_tokens_llm = pd.to_numeric(df_test_eval[llm_cols["output_tokens"]], errors="coerce").mean()
avg_cost_llm = compute_expected_cost(avg_input_tokens_llm, avg_output_tokens_llm)
avg_predicted_error_llm = error_cost(compute_expected_cost(llm_input_tokens, llm_output_tokens), llm_latency, avg_ql())

router_latency_series = pd.to_numeric(pd.Series([row["total_latency"] for row in router_performance_data]), errors="coerce")
router_input_tokens_series = pd.to_numeric(pd.Series([row["input_tokens"] for row in router_performance_data]), errors="coerce")
router_output_tokens_series = pd.to_numeric(pd.Series([row["output_tokens"] for row in router_performance_data]), errors="coerce")
router_error_series = pd.to_numeric(pd.Series([row["error_cost"] for row in router_performance_data]), errors="coerce")
router_correct_series = pd.to_numeric(pd.Series([row["correct_answer"] for row in router_performance_data]), errors="coerce")
router_num_big_decisions = (pd.Series([row["decision"] for row in router_performance_data]) == "big").sum()
router_q_series = pd.to_numeric(pd.Series([row["router_q"] for row in router_performance_data]), errors="coerce")
router_actual_cost_series = pd.to_numeric(pd.Series([row["actual_cost"] for row in router_performance_data]), errors="coerce")
router_predicted_cost_series = pd.to_numeric(pd.Series([row["predicted_cost"] for row in router_performance_data]), errors="coerce")
router_predicted_latency_series = pd.to_numeric(pd.Series([row["predicted_latency"] for row in router_performance_data]), errors="coerce")

router_decision_series = pd.Series([row["decision"] for row in router_performance_data])
router_is_big = (router_decision_series == "big")
router_is_small = (router_decision_series == "small")

avg_latency_router = router_latency_series.mean()
avg_input_tokens_router = router_input_tokens_series.mean()
avg_output_tokens_router = router_output_tokens_series.mean()
avg_cost_router = router_actual_cost_series.mean()
avg_error_router = router_error_series.mean()
avg_correct_answer_router = router_correct_series.mean()
avg_q_router = router_q_series.mean()
avg_predicted_cost = router_predicted_cost_series.mean()
avg_predicted_latency = router_predicted_latency_series.mean()
avg_predicted_error_router = error_cost(avg_predicted_cost, avg_predicted_latency, avg_q_router)

best_route_latency_series = pd.to_numeric(pd.Series([row["total_latency"] for row in best_route_performance_data]), errors="coerce")
best_route_input_tokens_series = pd.to_numeric(pd.Series([row["input_tokens"] for row in best_route_performance_data]), errors="coerce")
best_route_output_tokens_series = pd.to_numeric(pd.Series([row["output_tokens"] for row in best_route_performance_data]), errors="coerce")
best_route_error_series = pd.to_numeric(pd.Series([row["error_cost"] for row in best_route_performance_data]), errors="coerce")
best_route_correct_series = pd.to_numeric(pd.Series([row["correct_answer"] for row in best_route_performance_data]), errors="coerce")
best_route_num_big_decisions = (pd.Series([row["decision"] for row in best_route_performance_data]) == "big").sum()
best_route_actual_cost_series = pd.to_numeric(pd.Series([row["actual_cost"] for row in best_route_performance_data]), errors="coerce")

avg_latency_best_route = best_route_latency_series.mean()
avg_input_tokens_best_route = best_route_input_tokens_series.mean()
avg_output_tokens_best_route = best_route_output_tokens_series.mean()
avg_cost_best_route = best_route_actual_cost_series.mean()
avg_error_best_route = best_route_error_series.mean()
avg_correct_answer_best_route = best_route_correct_series.mean()

# Diagnostics: verify whether high router latency/cost comes from subset effects vs accounting bugs
router_big_share = router_is_big.mean()
router_small_share = router_is_small.mean()
router_latency_big = router_latency_series[router_is_big].mean()
router_latency_small = router_latency_series[router_is_small].mean()
router_cost_big = router_actual_cost_series[router_is_big].mean()
router_cost_small = router_actual_cost_series[router_is_small].mean()
router_tokens_in_big = router_input_tokens_series[router_is_big].mean()
router_tokens_in_small = router_input_tokens_series[router_is_small].mean()
router_tokens_out_big = router_output_tokens_series[router_is_big].mean()
router_tokens_out_small = router_output_tokens_series[router_is_small].mean()
reconstructed_latency = (router_big_share * router_latency_big) + (router_small_share * router_latency_small)
reconstructed_cost = (router_big_share * router_cost_big) + (router_small_share * router_cost_small)

# # Counterfactuals on exact routed subsets
# llm_latency_on_big_subset = pd.to_numeric(df_test_eval.loc[router_is_big.values, llm_cols["latency"]], errors="coerce").mean()
# slm_latency_on_small_subset = pd.to_numeric(df_test_eval.loc[router_is_small.values, slm_cols["latency"]], errors="coerce").mean()
# llm_cost_on_big_subset = compute_expected_cost(
#     pd.to_numeric(df_test_eval.loc[router_is_big.values, llm_cols["input_tokens"]], errors="coerce").mean(),
#     pd.to_numeric(df_test_eval.loc[router_is_big.values, llm_cols["output_tokens"]], errors="coerce").mean(),
# )

print("SLM Performance")
print(f"Selected SLM: {SLM_MODEL}")
print(f"Avg latency (test): {avg_latency_slm:.2f} seconds")
print(f"Avg latency (validation): {slm_latency:.2f} seconds")
print(f"Avg input tokens: {avg_input_tokens_slm:.2f}")
print(f"Avg output tokens: {avg_output_tokens_slm:.2f}")
print(f"Avg predicted error: {avg_predicted_error_slm:.4f}")

print("\nLLM Performance")
print(f"Selected LLM: {LLM_MODEL}")
print(f"Avg latency (test): {avg_latency_llm:.2f} seconds")
print(f"Avg latency (validation): {llm_latency:.2f} seconds")
print(f"Avg input tokens: {avg_input_tokens_llm:.2f}")
print(f"Avg output tokens: {avg_output_tokens_llm:.2f}")
print(f"Avg cost (test): {avg_cost_llm * MU_COST:.4f} Cents")
print(f"Avg cost (validation): {compute_expected_cost(llm_input_tokens, llm_output_tokens) * MU_COST:.4f} Cents")
print(f"Avg predicted error: {avg_predicted_error_llm:.2f}")
llm_correct_test = pd.to_numeric(df_test_eval[llm_cols["correct"]], errors="coerce").fillna(0)
print(f"Avg correct answer: {(llm_correct_test > 0.5).mean():.2f}")

print("\nRouter Performance")
print(f"Avg latency: {avg_latency_router:.2f} seconds")
print(f"Avg input tokens: {avg_input_tokens_router:.2f}")
print(f"Avg output tokens: {avg_output_tokens_router:.2f}")
print(f"Avg cost: {avg_cost_router * MU_COST:.4f} Cents")
print(f"Avg error: {avg_error_router:.2f}")
print(f"Avg predicted error: {avg_predicted_error_router:.2f}")
print(f"Avg correct answer: {avg_correct_answer_router:.4f}")
print(f"Number of big decisions: {router_num_big_decisions} / {len(router_performance_data)}")
print(f"Big share: {router_big_share:.4f}, Small share: {router_small_share:.4f}")
print(f"Latency by decision (small/big): {router_latency_small:.2f}s / {router_latency_big:.2f}s")
print(f"Cost by decision (small/big): {router_cost_small * MU_COST:.4f} / {router_cost_big * MU_COST:.4f} Cents")
print(f"Input tokens by decision (small/big): {router_tokens_in_small:.2f} / {router_tokens_in_big:.2f}")
print(f"Output tokens by decision (small/big): {router_tokens_out_small:.2f} / {router_tokens_out_big:.2f}")
print(f"Reconstructed avg latency/cost: {reconstructed_latency:.2f}s / {reconstructed_cost * MU_COST:.4f} Cents")
# print(f"LLM latency on routed-big subset: {llm_latency_on_big_subset:.2f}s")
# print(f"SLM latency on routed-small subset: {slm_latency_on_small_subset:.2f}s")
# print(f"LLM cost on routed-big subset: {llm_cost_on_big_subset * MU_COST:.4f} Cents")

print("\nBest Route Performance")
print(f"Avg latency: {avg_latency_best_route:.2f} seconds")
print(f"Avg input tokens: {avg_input_tokens_best_route:.2f}")
print(f"Avg output tokens: {avg_output_tokens_best_route:.2f}")
print(f"Avg cost: {avg_cost_best_route * MU_COST:.4f} Cents")
print(f"Avg error: {avg_error_best_route:.2f}")
print(f"Avg correct answer: {avg_correct_answer_best_route:.4f}")
print(f"Number of big decisions: {best_route_num_big_decisions} / {len(best_route_performance_data)}")


In [ ]:
# Plot E[C | L, x] and E[C | S, x] as continuous functions of p in [0, 1].
# Matches the plot style from test_router.ipynb, adapted for the regression router.


p_grid = np.linspace(0.0, 1.0, 1001)

q_l_grid = compute_ql(p_grid)
q_s_grid = compute_qs(p_grid)

llm_cost_mean = compute_expected_cost(llm_input_tokens, llm_output_tokens)

e_l = error_cost(
    llm_cost_mean,
    llm_latency,
    q_l_grid,
    lambd=LAMBDA_LATENCY,
    c_err=C_ERR,
)
e_s = error_cost(
    0.0,
    slm_latency,
    q_s_grid,
    lambd=LAMBDA_LATENCY,
    c_err=C_ERR,
)



cross_idx = int(np.argmin(np.abs(e_l - e_s)))

p_star = float(threshhold) if np.isfinite(threshhold) and 0 <= threshhold <= 1 else float(p_grid[cross_idx])
y_star = float(np.interp(p_star, p_grid, e_l))
p_threshold = float(np.clip(threshhold, 0.0, 1.0)) if np.isfinite(threshhold) else float(np.clip(p_star, 0.0, 1.0))

# Router decisions on test set.
if "test_probs" in globals() and len(test_probs) == len(df_test_eval):
    plot_test_probs = np.asarray(test_probs, dtype=float)
else:
    plot_test_probs = np.asarray(router_model.predict(X_test), dtype=float)

plot_features = {
    "has_indexed_vars": False,
    "is_simple_natural_language": False,
    "is_low_input": False,
}

test_points = []
for i, row in df_test_eval.iterrows():
    p_i = float(np.clip(plot_test_probs[i], 0.0, 1.0))
    decision_i = bayesian_decision_2path(p_i, plot_features, lambd=LAMBDA_LATENCY, cerr=C_ERR)

    slm_correct_row = int(_to_bool(row[slm_cols["correct"]]))
    llm_correct_row = int(_to_bool(row[llm_cols["correct"]]))

    slm_latency_i = pd.to_numeric(pd.Series([row[slm_cols["latency"]]]), errors="coerce").iloc[0]
    llm_latency_i = pd.to_numeric(pd.Series([row[llm_cols["latency"]]]), errors="coerce").iloc[0]
    llm_in_i = pd.to_numeric(pd.Series([row[llm_cols["input_tokens"]]]), errors="coerce").iloc[0]
    llm_out_i = pd.to_numeric(pd.Series([row[llm_cols["output_tokens"]]]), errors="coerce").iloc[0]

    if decision_i == "small":
        e_actual_i = float(error_cost(0.0, slm_latency_i, slm_correct_row, lambd=LAMBDA_LATENCY, c_err=C_ERR))
    else:
        llm_cost_i = compute_expected_cost(llm_in_i, llm_out_i)
        e_actual_i = float(error_cost(llm_cost_i, llm_latency_i, llm_correct_row, lambd=LAMBDA_LATENCY, c_err=C_ERR))

    test_points.append({
        "p": p_i,
        "decision": decision_i,
        "e_actual": e_actual_i,
        "dataset_name": row.get("dataset_name", "unknown"),
    })

test_points_df = pd.DataFrame(test_points)
test_points_df = test_points_df[np.isfinite(test_points_df["p"]) & np.isfinite(test_points_df["e_actual"])].copy()
test_small = test_points_df[test_points_df["decision"] == "small"]
test_big = test_points_df[test_points_df["decision"] == "big"]
test_small_left = test_small[test_small["p"] <= p_threshold].copy()
test_big_right = test_big[test_big["p"] >= p_threshold].copy()

plt.figure(figsize=(11, 5.5))
plt.plot(p_grid, e_s, "--", color="#b02dbb", linewidth=2.8, label="Expected Cost on val-split (small)", zorder=10)
plt.plot(p_grid, e_l, "--", color="#00b8f0", linewidth=2.8, label="Expected Cost on val-split (big)", zorder=10)

plt.scatter(test_big_right["p"], test_big_right["e_actual"], s=18, color="red", alpha=0.35, label="Actual Cost of Test points with p=correct_answer (big)", zorder=3)
plt.scatter(test_small_left["p"], test_small_left["e_actual"], s=18, color="green", alpha=0.35, label="Actual Cost of Test points with p=correct_answer (small)", zorder=3)

# Correlation lines are fit on full decision groups, then clipped at p_threshold for display.
if len(test_small) >= 2:
    coef_small = np.polyfit(test_small["p"].to_numpy(dtype=float), test_small["e_actual"].to_numpy(dtype=float), 1)
    p_line_small = np.linspace(0.0, p_threshold, 200)
    e_line_small = np.polyval(coef_small, p_line_small)
    r_small = float(np.corrcoef(test_small["p"].to_numpy(dtype=float), test_small["e_actual"].to_numpy(dtype=float))[0, 1])
    plt.plot(p_line_small, e_line_small, color="green", linewidth=2.0, alpha=0.95, label=f"Corr. Line on Test Split (small)", zorder=4)

if len(test_big) >= 2:
    coef_big = np.polyfit(test_big["p"].to_numpy(dtype=float), test_big["e_actual"].to_numpy(dtype=float), 1)
    p_line_big = np.linspace(p_threshold, 1.0, 200)
    e_line_big = np.polyval(coef_big, p_line_big)
    r_big = float(np.corrcoef(test_big["p"].to_numpy(dtype=float), test_big["e_actual"].to_numpy(dtype=float))[0, 1])
    plt.plot(p_line_big, e_line_big, color="red", linewidth=2.0, alpha=0.95, label=f"Corr. Line on Test Split (big)", zorder=4)


if np.isfinite(threshhold):
    plt.axvline(p_threshold, color="gray", linestyle=":", linewidth=1.3, alpha=0.9, label=f"Threshold {p_threshold:.4f}", zorder=11)

plt.xlim(0.0, 1.0)
plt.ylim(0.0, 22.0)
plt.xlabel("Predicted p value")
plt.ylabel("Expected Cost / Actual Cost")
plt.title(f"Expected Cost Curves vs Actual Cost on Test split")
plt.legend(frameon=False, loc="upper left")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# Same plot as above, but both model actual-cost point clouds span the full p range.
all_test_cost_points = []
for i, row in df_test_eval.iterrows():
    p_i = float(np.clip(plot_test_probs[i], 0.0, 1.0))

    slm_correct_row = int(_to_bool(row[slm_cols["correct"]]))
    llm_correct_row = int(_to_bool(row[llm_cols["correct"]]))

    slm_latency_i = pd.to_numeric(pd.Series([row[slm_cols["latency"]]]), errors="coerce").iloc[0]
    llm_latency_i = pd.to_numeric(pd.Series([row[llm_cols["latency"]]]), errors="coerce").iloc[0]
    llm_in_i = pd.to_numeric(pd.Series([row[llm_cols["input_tokens"]]]), errors="coerce").iloc[0]
    llm_out_i = pd.to_numeric(pd.Series([row[llm_cols["output_tokens"]]]), errors="coerce").iloc[0]

    llm_cost_i = compute_expected_cost(llm_in_i, llm_out_i)
    e_actual_small_i = float(error_cost(0.0, slm_latency_i, slm_correct_row, lambd=LAMBDA_LATENCY, c_err=C_ERR))
    e_actual_big_i = float(error_cost(llm_cost_i, llm_latency_i, llm_correct_row, lambd=LAMBDA_LATENCY, c_err=C_ERR))

    all_test_cost_points.append({
        "p": p_i,
        "e_actual_small": e_actual_small_i,
        "e_actual_big": e_actual_big_i,
        "dataset_name": row.get("dataset_name", "unknown"),
    })

all_test_cost_points_df = pd.DataFrame(all_test_cost_points)
all_test_cost_points_df = all_test_cost_points_df[
    np.isfinite(all_test_cost_points_df["p"])
    & np.isfinite(all_test_cost_points_df["e_actual_small"])
    & np.isfinite(all_test_cost_points_df["e_actual_big"])
].copy()

plt.figure(figsize=(11, 5.5))
plt.plot(p_grid, e_s, "--", color="#b02dbb", linewidth=2.8, label="Expected Cost on val-split (small)", zorder=10)
plt.plot(p_grid, e_l, "--", color="#00b8f0", linewidth=2.8, label="Expected Cost on val-split (big)", zorder=10)

plt.scatter(
    all_test_cost_points_df["p"],
    all_test_cost_points_df["e_actual_big"],
    s=18,
    color="red",
    alpha=0.35,
    label="Actual Cost of Test points with p=correct_answer (big, all p)",
    zorder=3,
)
plt.scatter(
    all_test_cost_points_df["p"],
    all_test_cost_points_df["e_actual_small"],
    s=18,
    color="green",
    alpha=0.35,
    label="Actual Cost of Test points with p=correct_answer (small, all p)",
    zorder=3,
)

p_line_full = np.linspace(0.0, 1.0, 200)
if len(all_test_cost_points_df) >= 2:
    coef_small_all = np.polyfit(
        all_test_cost_points_df["p"].to_numpy(dtype=float),
        all_test_cost_points_df["e_actual_small"].to_numpy(dtype=float),
        1,
    )
    e_line_small_all = np.polyval(coef_small_all, p_line_full)
    r_small_all = float(np.corrcoef(
        all_test_cost_points_df["p"].to_numpy(dtype=float),
        all_test_cost_points_df["e_actual_small"].to_numpy(dtype=float),
    )[0, 1])
    plt.plot(p_line_full, e_line_small_all, color="green", linewidth=2.0, alpha=0.95, label="Corr. Line on Test Split (small, all p)", zorder=4)

    coef_big_all = np.polyfit(
        all_test_cost_points_df["p"].to_numpy(dtype=float),
        all_test_cost_points_df["e_actual_big"].to_numpy(dtype=float),
        1,
    )
    e_line_big_all = np.polyval(coef_big_all, p_line_full)
    r_big_all = float(np.corrcoef(
        all_test_cost_points_df["p"].to_numpy(dtype=float),
        all_test_cost_points_df["e_actual_big"].to_numpy(dtype=float),
    )[0, 1])
    plt.plot(p_line_full, e_line_big_all, color="red", linewidth=2.0, alpha=0.95, label="Corr. Line on Test Split (big, all p)", zorder=4)

if np.isfinite(threshhold):
    plt.axvline(p_threshold, color="gray", linestyle=":", linewidth=1.3, alpha=0.9, label=f"Threshold {p_threshold:.4f}", zorder=11)

y_max = np.nanmax([
    np.nanmax(e_s),
    np.nanmax(e_l),
    all_test_cost_points_df["e_actual_small"].max(),
    all_test_cost_points_df["e_actual_big"].max(),
])

plt.xlim(0.0, 1.0)
plt.ylim(0.0, 22)
plt.xlabel("Predicted p value")
plt.ylabel("Expected Cost / Actual Cost")
plt.title(f"Expected Cost Curves vs Actual Cost on Test split (all test points)")
plt.legend(frameon=False, loc="upper left")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# Plot stacked histograms of the router's predicted probabilities by dataset/domain.
def plot_dataset_stacked_probability_hist(points_df, title, p_threshold, bins=30):
    plot_df = points_df[["p", "dataset_name"]].dropna().copy()
    plot_df["p"] = pd.to_numeric(plot_df["p"], errors="coerce").clip(0.0, 1.0)
    plot_df = plot_df.dropna(subset=["p"])

    domain_palette = {
        "MATH": "#1f77b4",
        "Medmcqa": "#ff7f0e",
        "Smiles": "#d62728",
    }
    domain_order = [name for name in domain_palette if name in set(plot_df["dataset_name"])]
    domain_order += sorted(set(plot_df["dataset_name"]) - set(domain_order))

    bin_edges = np.linspace(0.0, 1.0, bins + 1)
    plot_df["p_bin"] = pd.cut(plot_df["p"], bins=bin_edges, include_lowest=True, right=True)
    counts = (
        plot_df.groupby(["p_bin", "dataset_name"], observed=False)
        .size()
        .unstack(fill_value=0)
        .reindex(columns=domain_order, fill_value=0)
    )

    left_edges = bin_edges[:-1]
    width = bin_edges[1] - bin_edges[0]
    bottoms = np.zeros(len(counts), dtype=float)

    fig, ax = plt.subplots(figsize=(8, 4))
    for domain in domain_order:
        values = counts[domain].to_numpy(dtype=float)
        ax.bar(
            left_edges,
            values,
            width=width,
            bottom=bottoms,
            align="edge",
            color=domain_palette.get(domain, "#7f7f7f"),
            edgecolor="black",
            linewidth=0.4,
            label=domain,
        )
        bottoms += values
    ax.set_title(title)
    ax.set_xlabel("Predicted Probability of Complex (p)")
    ax.set_ylabel("Count")
    ax.set_xlim(0.0, 1.0)
    ax.legend(loc="upper left")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    plt.show()


# Test set: reuse test_points_df, which now carries dataset_name from df_test_eval.
plot_dataset_stacked_probability_hist(
    test_points_df,
    "Histogram of Regression Router Predicted Probabilities on Test Set",
    p_threshold,
)

# Validation set: attach dataset_name from the validation dataframe.
val_points_df = pd.DataFrame({
    "p": np.asarray(router_model.predict(X_val), dtype=float),
    "dataset_name": val_data.reset_index(drop=True)["dataset_name"],
})

plot_dataset_stacked_probability_hist(
    val_points_df,
    "Histogram of Regression Router Predicted Probabilities on Validation Set",
    p_threshold,
)


In [ ]:
# Predicted p-value histograms for the top 10 Smiles question templates.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

top_question_source_df = pd.read_csv("dataset/final_regressor/final_qwens_dataset.csv", low_memory=False)
top_question_source_df = top_question_source_df.loc[
    top_question_source_df["dataset_name"] == "Smiles"
].copy()
top_question_source_df["question_template"] = top_question_source_df["question"].str.replace(
    r"\s+SMILES:\s*.*$", "", regex=True
)

top_question_counts = top_question_source_df["question_template"].value_counts().head(10)
top_question_summary_df = (
    top_question_source_df.loc[
        top_question_source_df["question_template"].isin(top_question_counts.index)
    ]
    .groupby("question_template", as_index=False)
    .agg(
        Count=("question_template", "size"),
        avg_slm_accuracy_fraction=("slm_accuracy_fraction", "mean"),
    )
    .sort_values(["avg_slm_accuracy_fraction", "question_template"], ascending=[True, True])
)
top_questions = top_question_summary_df["question_template"].tolist()

test_plot_df = df_test_eval.reset_index(drop=True).copy()
test_plot_df["p"] = np.asarray(test_probs, dtype=float).clip(0.0, 1.0)
test_plot_df["split"] = "test"

val_plot_df = val_data.reset_index(drop=True).copy()
val_plot_df["p"] = np.asarray(y_pred_score_val, dtype=float).clip(0.0, 1.0)
val_plot_df["split"] = "validation"

question_p_df = pd.concat([test_plot_df, val_plot_df], ignore_index=True)
question_p_df = question_p_df.loc[question_p_df["dataset_name"] == "Smiles"].copy()
question_p_df["question_template"] = question_p_df["question"].str.replace(
    r"\s+SMILES:\s*.*$", "", regex=True
)
question_p_df = question_p_df.loc[question_p_df["question_template"].isin(top_questions)].copy()

def plot_top_question_p_hist(points_df, split, title, p_threshold, bins=30):
    plot_df = points_df.loc[points_df["split"] == split, ["p"]].dropna().copy()
    plot_df["p"] = pd.to_numeric(plot_df["p"], errors="coerce").clip(0.0, 1.0)
    plot_df = plot_df.dropna(subset=["p"])

    bin_edges = np.linspace(0.0, 1.0, bins + 1)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(
        plot_df["p"],
        bins=bin_edges,
        color="#4c78a8",
        edgecolor="none",
        linewidth=0,
        alpha=0.9,
    )
    ax.set_title(title)
    ax.set_xlabel("Predicted p")
    ax.set_ylabel("Count")
    ax.set_xlim(0.0, 1.0)
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    plt.show()


plot_top_question_p_hist(
    question_p_df,
    "test",
    "Predicted p Distribution for Top 10 Smiles Question Templates on Test Set",
    p_threshold,
)

plot_top_question_p_hist(
    question_p_df,
    "validation",
    "Predicted p Distribution for Top 10 Smiles Question Templates on Validation Set",
    p_threshold,
)


In [ ]:
# Per-question predicted p-value histograms for the top 10 Smiles question templates.
# Run the aggregate top-question histogram cell above first so question_p_df exists.
top_question_source_df = pd.read_csv("dataset/final_regressor/final_qwens_dataset.csv", low_memory=False)
top_question_source_df = top_question_source_df.loc[
    top_question_source_df["dataset_name"] == "Smiles"
].copy()
top_question_source_df["question_template"] = top_question_source_df["question"].str.replace(
    r"\s+SMILES:\s*.*$", "", regex=True
)
top_question_source_df["slm_accuracy_fraction"] = pd.to_numeric(
    top_question_source_df["slm_accuracy_fraction"], errors="coerce"
)

top_question_counts = top_question_source_df["question_template"].value_counts().head(10)
top_question_summary_df = (
    top_question_source_df.loc[
        top_question_source_df["question_template"].isin(top_question_counts.index)
    ]
    .groupby("question_template", as_index=False)
    .agg(
        Count=("question_template", "size"),
        avg_slm_accuracy_fraction=("slm_accuracy_fraction", "mean"),
    )
    .sort_values(["avg_slm_accuracy_fraction", "question_template"], ascending=[True, True])
)
ordered_top_questions = top_question_summary_df["question_template"].tolist()

bins = np.linspace(0.0, 1.0, 21)
fig, axes = plt.subplots(5, 2, figsize=(14, 18), sharex=False, sharey=False)
axes = axes.ravel()

for ax, question in zip(axes, ordered_top_questions):
    question_values = question_p_df.loc[
        question_p_df["question_template"] == question,
        "p",
    ].dropna()

    ax.hist(
        question_values,
        bins=bins,
        color="#4c78a8",
        edgecolor="none",
        linewidth=0,
        alpha=0.9,
    )
    ax.set_title(question, fontsize=9)
    ax.set_xlabel("Predicted p")
    ax.set_ylabel("Count")
    ax.set_xlim(0.0, 1.0)
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("Predicted p Distribution by Top Smiles Question Template", y=0.995)
plt.tight_layout()
plt.show()
